# N-NBCF_ITR


In [1]:
import numpy as np
import math
from math import prod
from decimal import Decimal, getcontext
import pickle
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import json
import os
getcontext().prec = 5

## Pembuatan matriks rating

In [2]:
# inisialisasi dataset
kolom = ['user_id', 'movie_id', 'rating', 'timestamp']
# kolom = ['user_id', 'movie_id', 'rating']
df = pd.read_csv('../../Dataset/ml-100k/u.data', sep='\t', names=kolom)

# Menggunakan path absolut
# df = pd.read_csv('../../Dataset/toy_rating.csv', sep=';')
print(df.head())
# pembuatan matriks rating
n_users = df.user_id.nunique()
n_items = df.movie_id.nunique()
rating = np.zeros((n_users, n_items))
for row in df.itertuples():
    rating[row[1]-1, row[2]-1] = row[3]

# print('Jumlah pengguna:', n_users)
# print('Jumlah item:', n_items)
# print('Jumlah rating:', np.count_nonzero(rating))
# print('ukuran matriks rating:', rating.shape)
print('Contoh matriks rating:\n', rating[:20, :20])

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
Contoh matriks rating:
 [[5. 3. 4. 3. 3. 5. 4. 1. 5. 3. 2. 5. 5. 5. 5. 5. 3. 4. 5. 4.]
 [4. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 4. 4. 0. 0. 0. 0. 3. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 4. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [4. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 4. 0. 0. 0.]
 [4. 0. 0. 0. 0. 0. 2. 4. 4. 0. 0. 4. 2. 5. 3. 0. 0. 0. 4. 0.]
 [0. 0. 0. 5. 0. 0. 5. 5. 5. 4. 3. 5. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 3. 0. 0. 0. 3. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 5. 4. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [4. 0. 0. 4. 0. 0. 4. 0. 4. 0. 4. 5. 3. 0. 0. 4. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 4. 5. 0. 2. 2. 0. 0. 5. 0. 0. 0. 0. 0.]
 [0. 0. 0. 5. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.

## Probabilitas Prior

In [3]:
def prior_prob(rating, target_user=None, target_item=None, alpha=0.01, R=5):
    b_user = len(rating)
    b_item = len(rating[0])
    kelas_rating = list(range(1, R+1))

    if target_user is None:
        target_user = list(range(b_user))
        print("target user", target_user)
    if target_item is None:
        target_item = list(range(b_item))
        print("target item", target_item)

    prior_ub = np.zeros((R, len(target_item)))  #
    prior_ib = np.zeros((R, len(target_user)))

    item_map_prior = {item_idx: i for i, item_idx in enumerate(target_item)}
    user_map_prior = {user_idx: i for i, user_idx in enumerate(target_user)}

    for y in kelas_rating:
        y_index = y - 1

        # prior user-based
        for j in target_item:   
            rating_item = rating[:, j]
            non_zero_ratings = rating_item[rating_item != 0]
            len_nonzero = len(non_zero_ratings)
            val = 0.0
            count_y = np.sum(non_zero_ratings == y)  # menghitung jumlah rating yang sama dengan y untuk item j

            if len_nonzero > 0:
                val = (count_y + alpha) / (len_nonzero + alpha * R)
            else:
                val = alpha / (alpha * R)
            prior_ub[y_index][item_map_prior[j]] = val
        
        # prior item-based
        for u in target_user:
            rating_user = rating[u, :]
            non_zero_ratings = rating_user[rating_user != 0]
            len_nonzero = len(non_zero_ratings)
            count_y = np.sum(non_zero_ratings == y)
            val = 0.0
            
            if len_nonzero > 0:
                val = (count_y + alpha) / (len_nonzero + alpha * R)
            else:
                val = alpha / (alpha * R)
            prior_ib[y_index][user_map_prior[u]] = val
    return prior_ub, prior_ib

In [4]:
# R = 5
# b_item = len(rating[0])
# target_item = list(range(b_item))
# prior_ub = np.zeros((R, len(target_item)))
# print(prior_ub)

In [5]:
prior = prior_prob(rating)
prior_ub = prior[0]
prior_ib = prior[1]
print('Prior User-Based:\n', prior_ub)
print('Prior Item-Based:\n', prior_ib.shape)

target user [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 21

In [6]:
prior__ = np.load('hasil_eval_ndcg/prior/fold_1/prior_ub.npy')
print('Prior User-Based dari file:\n', prior__)

Prior User-Based dari file:
 [[0.01830048 0.07624941 0.13337775 ... 0.00952381 0.00952381 0.00952381]
 [0.06007049 0.12384579 0.23997335 ... 0.96190476 0.00952381 0.00952381]
 [0.21148675 0.41894336 0.2532978  ... 0.00952381 0.96190476 0.96190476]
 [0.43077927 0.30471204 0.23997335 ... 0.00952381 0.00952381 0.00952381]
 [0.27936301 0.07624941 0.13337775 ... 0.00952381 0.00952381 0.00952381]]


## ITR

### Triangle'

In [7]:
from itertools import combinations
rating_ub = rating
b_user = len(rating_ub)
matrix_ub = np.eye(b_user)
mask_ub = [rating_ub[row, :] != 0 for row in range(b_user)]
print(mask_ub[0][:10])
print(mask_ub[4][:10])
count=0
for u, v in combinations(range(b_user), 2):
    common_items = mask_ub[u] | mask_ub[v]
    if np.sum(common_items) == 0:
        continue
    if u == 0 and v == 4:
        u_ci = rating_ub[u, common_items]
        v_ci = rating_ub[v, common_items]
        print(u, v, common_items[:10], np.sum(common_items))


[ True  True  True  True  True  True  True  True  True  True]
[ True  True False False False False False False False False]
0 4 [ True  True  True  True  True  True  True  True  True  True] 367


In [8]:
from itertools import combinations

def triangle(rating, base='user'):
    if base == 'user':
        rating_ub = rating
        b_user = len(rating_ub)
        matrix_ub = np.eye(b_user)
        # print('awal',matrix_ub)
        mask_ub = [rating_ub[row, :] != 0 for row in range(b_user)]
        # print(len(mask_ub[0]))
        # jumlah_loop = 0

        for u, v in combinations(range(b_user), 2):
            common_items = mask_ub[u] | mask_ub[v] # I_uv
            # jumlah_loop += 1
            # print(common_items)  
            if np.sum(common_items) == 0:
                continue
            u_ci = rating_ub[u, common_items]  #r_ui
            # print(u, u_ci)
            v_ci = rating_ub[v, common_items]  #r_vi
            # print(v, v_ci)

            pembilang = np.sqrt(np.sum((u_ci - v_ci) ** 2))
            penyebut = np.sqrt(np.sum(u_ci ** 2)) + np.sqrt(np.sum(v_ci ** 2))

            if penyebut != 0:
                triangle = 1 - pembilang / penyebut
            else:
                triangle = 0

            matrix_ub[u, v] = triangle  # isi segitiga atas
            matrix_ub[v, u] = triangle  # isi segitiga bawah
        # print('Setelah update: ', matrix_ub)
        # print(jumlah_loop)
        return matrix_ub
    
    elif base == 'item':
        rating_ib = rating.T
        b_item = len(rating_ib)
        matrix_ib = np.eye(b_item)
        mask_ib = [rating_ib[row, :] != 0 for row in range(b_item)]

        for i,j in combinations(range(b_item), 2):
            common_users = mask_ib[i] | mask_ib[j]
            if np.sum(common_users) == 0:
                continue

            i_cu = rating_ib[i, common_users]
            j_cu = rating_ib[j, common_users]

            pembilang = np.sqrt(np.sum((i_cu - j_cu) ** 2))
            penyebut = np.sqrt(np.sum(i_cu ** 2)) + np.sqrt(np.sum(j_cu ** 2))

            if penyebut != 0:
                triangle = 1 - pembilang / penyebut
            else:
                triangle = 0

            matrix_ib[i, j] = triangle
            matrix_ib[j, i] = triangle

        return matrix_ib

In [9]:
# triangle_user = triangle(rating, base='user')
# triangle_item = triangle(rating, base='item')

In [10]:
# print(triangle_user[:6, :6])

In [11]:
# print(triangle_item[:6, :6])

## URP

### mean rating

In [12]:
def compute_mean_rating_user(rating):
    mean_user = np.ma.masked_equal(rating, 0).mean(axis=1)
    return mean_user
def compute_mean_rating_item(rating):
    mean_item = np.ma.masked_equal(rating, 0).mean(axis=0)
    return mean_item

In [13]:
# mean_user = compute_mean_rating_user(rating)
# mean_item = compute_mean_rating_item(rating)

In [14]:
# # print(rating)
# print(mean_item[:50])

### mean centered rating

In [15]:
def compute_mean_centered_rating_user(rating):
    non_zero_ratings = np.ma.masked_equal(rating, 0)
    centered_rating = non_zero_ratings - mean_user[:, np.newaxis]
    return centered_rating.filled(0)
def compute_mean_centered_rating_item(rating):
    non_zero_ratings = np.ma.masked_equal(rating, 0)
    centered_rating = non_zero_ratings - mean_item[np.newaxis, :]
    return centered_rating.filled(0)

In [16]:
# mcru = compute_mean_centered_rating_user(rating)
# mcri = compute_mean_centered_rating_item(rating)

In [17]:
# print(mcri)

### standar deviasi

In [18]:
def standar_deviasi_user(rating, mcru):
    b_user = len(mcru)
    # print(b_user)
    sd_user = np.zeros((b_user, ))
    # print(sd_user)
    for u in range(b_user):
        mcu = mcru[u, :]
        rating_user = rating[u, :]
        # print(mcu)
        non_zero_mcu = mcu[rating_user != 0]
        non_zero_rating = rating_user[rating_user != 0]
        non_zero_len = len(non_zero_rating)
        if non_zero_len > 0:
            sd_user[u] = np.sqrt(np.sum(non_zero_mcu ** 2) / non_zero_len)
    return sd_user

def standar_deviasi_item(rating, mcri):
    b_item = len(mcri[0])
    sd_item = np.zeros((b_item, ))
    # print(sd_item)
    for i in range(b_item):
        mci = mcri[:, i]
        rating_item = rating[:, i]
        non_zero_mci = mci[rating_item != 0]
        non_zero_rating = rating_item[rating_item != 0]
        non_zero_len = len(non_zero_rating)
        # print(non_zero_len)
        if non_zero_len > 0:
            sd_item[i] = np.sqrt(np.sum(non_zero_mci ** 2) / non_zero_len)
    return sd_item

In [19]:
# sdu = standar_deviasi_user(rating, mcru)
# sdi = standar_deviasi_item(rating, mcri)

In [20]:
# print(sdi[:50])

### menghitung URP

In [21]:
from itertools import combinations
def URP(rating, mean_user, mean_item, sdu, sdi, base='user'):
    if base == 'user':
        user_count = len(rating) # 7
        matrix_ub = np.eye(user_count)  # 7x7

        for u, v in combinations(range(user_count), 2):
            if not (mean_user[u] and mean_user[v] and sdu[u] and sdu[v]):
                continue
           
            u_ci_mr = mean_user[u]
            v_ci_mr = mean_user[v]
            u_ci_sd = sdu[u]
            v_ci_sd = sdu[v]

            pembilang = 1
            penyebut = 1 + math.exp(-abs((u_ci_mr - v_ci_mr)) * abs((u_ci_sd - v_ci_sd)))

            if penyebut != 0:
                matrix_ub[u, v] = 1 - (pembilang / penyebut)
                matrix_ub[v, u] = matrix_ub[u, v]
            else:
                matrix_ub[u, v] = 0
                matrix_ub[v, u] = 0
        return matrix_ub

    elif base == 'item':
        item_count = len(rating[0])
        matrix_ib = np.eye(item_count)

        for i, j in combinations(range(item_count), 2):
            if not (mean_item[i] and mean_item[j] and sdi[i] and sdi[j]):
                continue
            i_cu_mr = mean_item[i]
            j_cu_mr = mean_item[j]
            i_cu_sd = sdi[i]
            j_cu_sd = sdi[j]

            pembilang = 1
            penyebut = 1 + math.exp(-abs((i_cu_mr - j_cu_mr)) * abs((i_cu_sd - j_cu_sd)))

            if penyebut != 0:
                matrix_ib[i, j] = 1 - (pembilang / penyebut)
                matrix_ib[j, i] = matrix_ib[i, j]
            else:
                matrix_ib[i, j] = 0
                matrix_ib[j, i] = 0
        return matrix_ib


In [22]:
# user_count = len(rating) # 7
# for u, v in combinations(range(user_count), 2):
#     if not (mean_user[u] and mean_user[v] and sdu[u] and sdu[v]):
#         continue
    
#     u_ci_mr = mean_user[u]
#     v_ci_mr = mean_user[v]
#     u_ci_sd = sdu[u]
#     v_ci_sd = sdu[v]
#     print(u_ci_mr - v_ci_mr)

In [23]:
# urp_user = URP(rating, mean_user, mean_item, sdu, sdi, base='user')
# urp_item = URP(rating, mean_user, mean_item, sdu, sdi, base='item')

In [24]:
# print(urp_user[:6, :6])

In [25]:
# print(urp_item[:6, :6])

### menghitung ITR

In [26]:
def ITR(rating, triangle_user, triangle_item, urp_user, urp_item, base='user'):
    if base == 'user':
        b_user = len(rating)
        matrix_ub = np.eye(b_user)
        for u, v in combinations(range(b_user), 2):
            matrix_ub[u, v] = triangle_user[u, v] * urp_user[u, v]
            matrix_ub[v, u] = matrix_ub[u, v]
        return matrix_ub
    elif base == 'item':
        b_item = len(rating[0])
        matrix_ib = np.eye(b_item)
        for i, j in combinations(range(b_item), 2):
            matrix_ib[i, j] = triangle_item[i,j] * urp_item[i,j]
            matrix_ib[j ,i] = matrix_ib[i, j]
        return matrix_ib

In [27]:
# ITR_user = ITR(rating, triangle_user, triangle_item, urp_user, urp_item, base='user')
# ITR_item = ITR(rating, triangle_user, triangle_item, urp_user, urp_item, base='item')

In [28]:
# print(ITR_user)

In [29]:
# print(ITR_item[:150, :150])

In [30]:
# toy = pd.read_csv('../../Dataset/toy_similaritas.csv', sep=';', header=None).values
# toy = toy.astype(ITR_user.dtype)
# print(toy)

## mencari tetangga

In [31]:
def tuku_user(rating, ITR_user, user_id, item_id, k=20):
    b_user = rating.shape[0]
    user_similarities = ITR_user[user_id, :]
    # print(user_similarities)
    tuku = []
    for v in range(b_user):
        if v != user_id and rating[v, item_id] != 0:
            tuku.append((user_similarities[v], v))
    tuku.sort(key=lambda x: (-x[0], x[1]))
    return [n for n in tuku if n[0] > 0][:k]
def tuku_item(rating, ITR_item, user_id, item_id, k=20):
    b_item = rating.shape[1]
    item_similarities = ITR_item[item_id, :]
    tuku = []
    for j in range(b_item):
        if j != item_id and rating[user_id, j] != 0:
            tuku.append((item_similarities[j], j))
    tuku.sort(key=lambda x: (-x[0], x[1]))
    return [n for n in tuku if n[0] > 0][:k]


In [32]:
# t_user = tuku_item(rating, ITR_user, user_id=2, item_id=3, k=4)
# print(t_user)

In [33]:
# def sort_tetangga(rating, similaritas, user, item, k=2):
#     dt = rating[:, item]
#     simi = similaritas[user, :]
#     idx_data = np.where(dt != 0)
#     nilai_sim = simi[idx_data]
#     print(nilai_sim)
#     idx_sim = (-nilai_sim).argsort()[:k]
#     print(idx_sim)

In [34]:
# sort_tetangga(rating, toy, user=2, item=3, k=4)

In [35]:
# t_item = tuku_item(rating, ITR_item, user_id=2, item_id=3, k=1)
# print(t_item)

## Likelihood dengan tetangga

In [36]:
from numba import njit

# tetangga_user = np.array([n[1] for n in t_user], dtype=np.int32)
# tetangga_item = np.array([n[1] for n in t_item], dtype=np.int32)

@njit
def compute_likelihood_user(rating, u, i, y, tetangga_item, alpha=0.01, R=5):
    if len(tetangga_item) == 0:
        return 1.0
    product = 1.0
    u_with_y = np.where(rating[:, i] == y)[0]  # mengetahui r_v,i = y

    for j in tetangga_item:
        k = rating[u, j]  # r_u,j != 0 mengetahui r_u,i = y
        kondisi_pembilang = 0.0
        kondisi_penyebut = 0.0
        for v in u_with_y:
            if rating[v, j] != 0:
                kondisi_penyebut += 1.0
                if rating[v, j] == k:
                    kondisi_pembilang += 1.0
        probabilitas = (kondisi_pembilang + alpha) / (kondisi_penyebut + (R * alpha))
        product *= probabilitas
    return product

@njit
def compute_likelihood_item(rating, u, i, y, tetangga_user, alpha=0.01, R=5):
    if len(tetangga_user) == 0:
        return 1.0
    product = 1.0
    i_with_y = np.where(rating[u, :] == y)[0]

    for v in tetangga_user:
        k = rating[v, i]
        kondisi_pembilang = 0.0
        kondisi_penyebut = 0.0
        for j in i_with_y:
            if rating[v, j] != 0:
                kondisi_penyebut += 1.0
                if rating[v, j] == k:
                    kondisi_pembilang += 1.0
        probabilitas = (kondisi_pembilang + alpha) / (kondisi_penyebut + (R * alpha))
        product *= probabilitas
    return product

In [37]:
# li_u = compute_likelihood_user(rating, u=2, i=0, y=1, tetangga_item=tetangga_item, alpha=0.01, R=5)
# print("Likelihood User:", li_u)

In [38]:
# li_i = compute_likelihood_item(rating, u=2, i=0, y=1, tetangga_user=tetangga_user, alpha=0.01, R=5)
# print("Likelihood Item:", li_i)

In [39]:
# print(tetangga_user)

## Prediksi rating


In [40]:
def contoh_prediksi_rating(rating, user, item, prior_ub, prior_ib, ITR_user, ITR_item, k_user=20, k_item=20, R=5):
    t_user = tuku_user(rating, ITR_user, user, item, k_user)
    t_item = tuku_item(rating, ITR_item, user, item, k_item)

    tetangga_ub = np.array([n[1] for n in t_user], dtype=np.int32)
    tetangga_ib = np.array([n[1] for n in t_item], dtype=np.int32)

    score_hybrid = []
    detail_y = []

    for y in range(1, R+1):
        prior_u = prior_ub[y-1][item]
        prior_i = prior_ib[y-1][user]

        li_u = compute_likelihood_user(rating, user, item, y, tetangga_ib, alpha=0.01, R=5)
        li_i = compute_likelihood_item(rating, user, item, y, tetangga_ub, alpha=0.01, R=5)

        posterior_ub = prior_u * li_u
        posterior_ib = prior_i * li_i

        score_ub = (posterior_ub) ** (1 / (k_item + 1)) if k_item > 0 else -100
        score_ib = (posterior_ib) ** (1 / (k_user + 1)) if k_user > 0 else -100

        hybrid_score = score_ub * score_ib
        score_hybrid.append(hybrid_score)
        detail_y.append({
            'rating_value' : y,
            'prior_user' : prior_u,
            'prior_item' : prior_i,
            'likelihood_user' : li_u,
            'likelihood_item' : li_u,
            'posterior_user' : posterior_ub,
            'posterior_item' : posterior_ib,
            'score_user' : score_ub,
            'score_item' : score_ib,
            'hybrid_score' : hybrid_score
        })
        
    max_score = max(score_hybrid)
    idx = [i for i, score in enumerate(score_hybrid) if np.isclose(score, max_score)]
    if len(idx) == 1:
        prediksi_rating = idx[0] + 1
    elif len(idx) > 1:
        Z = sum(score_hybrid)
        product = 0.0
        for idx_rating, score in enumerate(score_hybrid, start=1):
            normalisasi = score / Z
            prob = normalisasi * idx_rating
            product += prob
            prediksi_rating = product
    return prediksi_rating, score_hybrid, detail_y

In [41]:
def prediksi_rating(rating, user, item, user_map, item_map, prior_ub, prior_ib, ITR_user, ITR_item, prior_u_map=None, prior_i_map=None, k_user=20, k_item=20, R=5):
    user_global = user - 1  
    item_global = item - 1

    t_user = tuku_user(rating, ITR_user, user_global, item_global, k_user)
    t_item = tuku_item(rating, ITR_item, user_global, item_global, k_item)

    tetangga_ub = np.array([n[1] for n in t_user], dtype=np.int32)
    tetangga_ib = np.array([n[1] for n in t_item], dtype=np.int32)

    user_dense = user_map.get(user)
    item_dense = item_map.get(item)

    if user_dense is None or item_dense is None:
        return None, [], []
    
    if prior_u_map is not None:
        if item_dense in prior_u_map:
            idx_prior_user = prior_u_map[item_dense]
        else:
            return None, [], []
    
    if prior_i_map is not None:
        if user_dense in prior_i_map:
            idx_prior_item = prior_i_map[user_dense]
        else:
            return None, [], []

    score_hybrid = []
    detail_y = []

    for y in range(1, R+1):
        prior_u = prior_ub[y-1][idx_prior_user]
        prior_i = prior_ib[y-1][idx_prior_item]

        li_u = compute_likelihood_user(rating, user_global, item_global, y, tetangga_ib, alpha=0.01, R=5)
        li_i = compute_likelihood_item(rating, user_global, item_global, y, tetangga_ub, alpha=0.01, R=5)

        posterior_ub = prior_u * li_u
        posterior_ib = prior_i * li_i

        score_ub = (posterior_ub) ** (1 / (k_item + 1)) if k_item > 0 else -100
        score_ib = (posterior_ib) ** (1 / (k_user + 1)) if k_user > 0 else -100

        hybrid_score = score_ub * score_ib
        score_hybrid.append(hybrid_score)
        detail_y.append({
            'rating_value' : y,
            'prior_user' : prior_u,
            'prior_item' : prior_i,
            'likelihood_user' : li_u,
            'likelihood_item' : li_i,
            'posterior_user' : posterior_ub,
            'posterior_item' : posterior_ib,
            'score_user' : score_ub,
            'score_item' : score_ib,
            'hybrid_score' : hybrid_score
        })
        
    max_score = max(score_hybrid)
    idx = [i for i, score in enumerate(score_hybrid) if np.isclose(score, max_score)]

    if len(idx) == 1:
        prediksi_rating = idx[0] + 1
    elif len(idx) > 1:
        Z = sum(score_hybrid)
        product = 0.0
        for idx_rating, score in enumerate(score_hybrid, start=1):
            normalisasi = score / Z
            prob = normalisasi * idx_rating
            product += prob
            prediksi_rating = product
    return prediksi_rating, score_hybrid, detail_y

In [42]:
def prediksi_rating_EV(rating, user, item, user_map, item_map, prior_ub, prior_ib, ITR_user, ITR_item, prior_u_map=None, prior_i_map=None, k_user=20, k_item=20, R=5):
    user_global = user - 1  
    item_global = item - 1

    t_user = tuku_user(rating, ITR_user, user_global, item_global, k_user)
    t_item = tuku_item(rating, ITR_item, user_global, item_global, k_item)

    tetangga_ub = np.array([n[1] for n in t_user], dtype=np.int32)
    tetangga_ib = np.array([n[1] for n in t_item], dtype=np.int32)

    user_dense = user_map.get(user)
    item_dense = item_map.get(item)

    if user_dense is None or item_dense is None:
        return None, [], []
    
    if prior_u_map is not None:
        if item_dense in prior_u_map:
            idx_prior_user = prior_u_map[item_dense]
        else:
            return None, [], []
    
    if prior_i_map is not None:
        if user_dense in prior_i_map:
            idx_prior_item = prior_i_map[user_dense]
        else:
            return None, [], []

    score_hybrid = []
    detail_y = []

    for y in range(1, R+1):
        prior_u = prior_ub[y-1][idx_prior_user]
        prior_i = prior_ib[y-1][idx_prior_item]

        li_u = compute_likelihood_user(rating, user_global, item_global, y, tetangga_ib, alpha=0.01, R=5)
        li_i = compute_likelihood_item(rating, user_global, item_global, y, tetangga_ub, alpha=0.01, R=5)

        posterior_ub = prior_u * li_u
        posterior_ib = prior_i * li_i

        score_ub = (posterior_ub) ** (1 / (k_item + 1)) if k_item > 0 else -100
        score_ib = (posterior_ib) ** (1 / (k_user + 1)) if k_user > 0 else -100

        hybrid_score = score_ub * score_ib
        score_hybrid.append(hybrid_score)
        detail_y.append({
            'rating_value' : y,
            'prior_user' : prior_u,
            'prior_item' : prior_i,
            'likelihood_user' : li_u,
            'likelihood_item' : li_i,
            'posterior_user' : posterior_ub,
            'posterior_item' : posterior_ib,
            'score_user' : score_ub,
            'score_item' : score_ib,
            'hybrid_score' : hybrid_score
        })
        
    
    Z = sum(score_hybrid)
    product = 0.0
    for idx_rating, score in enumerate(score_hybrid, start=1):
        normalisasi = score / Z
        prob = normalisasi * idx_rating
        product += prob
        prediksi_rating = product
    return prediksi_rating, score_hybrid, detail_y

In [43]:
# pe = contoh_prediksi_rating(rating, 2, 0, prior_ub, prior_ib, ITR_user, ITR_item, k_user=1, k_item=1, R=5)
# print(pe)

In [44]:
# R = 5
# for y in range(1, R+1):
#     p = prior_ub[y-1][0]
#     print("y:", y, "prior_ub:", p)

## TOP N Rekomendasi


In [45]:
def contoh_Top_N(rating, user_idx, ITR_user, ITR_item, prior_ub, prior_ib, N=3, k_user=20, k_item=20):
    # item yang belum dirating user
    unrated_items = np.where(rating[user_idx, :] == 0)[0]
    print("Unrated items:", unrated_items)

    results = []
    for item_idx in unrated_items:
        pred = contoh_prediksi_rating(
            rating, user_idx, item_idx, prior_ub, prior_ib, ITR_user, ITR_item, k_user=k_user, k_item=k_item, R=5
        )[0]
        results.append((pred, item_idx))
        print(results)

    # urutkan dari prediksi tertinggi
    results.sort(key=lambda x: (-x[0], x[1]))
    return results[:N]

In [46]:
# top_n = contoh_Top_N(rating, user_idx=2, ITR_user=ITR_user, ITR_item=ITR_item, prior_ub=prior_ub, prior_ib=prior_ib, N=3, k_user=2, k_item=2)


In [47]:
# test_toy = pd.read_csv('../../Dataset/test_toy.csv', sep=';')
# # print(test_toy)

# n_users = test_toy.user_id.nunique()
# n_items = test_toy.item_id.nunique()
# test = np.zeros((n_users, n_items))
# for row in test_toy.itertuples():
#     test[row[1]-1, row[2]-1] = row[3]

# print(rating)
# print(test)

In [48]:
def hitung_idcg(ground_truth, top_n, N):
    """Hitung DCG@N untuk relevansi biner (item ada di ground_truth)."""
    ground_truth_set = set(ground_truth)
    dcg = 0.0

    for rank, entry in enumerate(top_n[:N], start=1):
        if isinstance(entry, (list, tuple)):
            # Format contoh_Top_N: (prediksi, item_idx)
            item_id = entry[1]
        else:
            item_id = entry

        pembilang = 1.0 
        if pembilang:
            dcg += pembilang / math.log2(rank + 1)

    return dcg

In [49]:
def calculate_dcg(predicted_items, ground_truth, n):
    dcg = 0.0
    # Hanya mengambil item sampai posisi n
    top_n_predicted = predicted_items[:n]

    for i, item in enumerate(top_n_predicted):
        rank = i + 1  # Posisi dimulai dari 1
        # Cek apakah item ada di ground truth (relevansi biner)
        # Rumus: rel_i / log2(rank + 1)
        dcg += (1 / math.log2(rank + 1)) * 1 if item in ground_truth else 0

    return dcg

In [50]:
# p = [item[1] for item in top_n]
# print("Predicted Top-N items:", p)
# ground_truth = test[2, :].nonzero()[0]  
# print("Ground Truth:", ground_truth)
# # idcg = hitung_idcg(ground_truth, top_n, N=3)
# # print("IDCG@3:", idcg)

In [51]:
# dcg_cal = calculate_dcg([item[1] for item in top_n], ground_truth, n=4)
# print("DCG@3 (calculate_dcg):", dcg_cal)

In [52]:
# n = rating[2, :] == 0
# unrated_item = np.where(n)[0]
# print(unrated_item)

In [53]:
# prior = np.load('hasil_eval_ndcg/prior/fold_1/prior_ib.npy')
# print(len(prior[1]))

## Uji Coba

In [54]:
def load_data(fold):
    folder_base = '../../Dataset/ml-100k/'
    data_train = f'{folder_base}u{fold}.base'
    data_test = f'{folder_base}u{fold}.test'
    data_global = f'{folder_base}u.data'

    nama_kolom = ['user_id', 'item_id', 'rating', 'timestamp']

    # ambil banyak user dan item dari data global
    read_global = pd.read_csv(data_global, sep='\t', names=nama_kolom)
    u_global = read_global['user_id'].nunique()
    i_global = read_global['item_id'].nunique()

    # load data train dan test
    read_train = pd.read_csv(data_train, sep='\t', names=nama_kolom)
    read_test = pd.read_csv(data_test, sep='\t', names=nama_kolom)

    # matriks rating data global untuk ITR
    mr_global = np.zeros((u_global, i_global))
    # memasukkan data train ke matriks rating global
    for row in read_train.itertuples():
        mr_global[row.user_id-1, row.item_id-1] = row.rating

    # matriks rating data train untuk prior
    user_fold = np.unique(np.concatenate((read_train['user_id'].unique(), read_test['user_id'].unique())))
    item_fold = np.unique(np.concatenate((read_train['item_id'].unique(), read_test['item_id'].unique())))

    # maping user dan item di user_fold dan item_fold ke indeks matriks rating fold
    user_map = {user_id: idx for idx, user_id in enumerate(user_fold)}
    item_map = {item_id: idx for idx, item_id in enumerate(item_fold)}

    b_user_dense = len(user_fold)
    b_item_dense = len(item_fold)

    mr_dense = np.zeros((b_user_dense, b_item_dense))
    for row in read_train.itertuples():
        map_u = user_map[row.user_id]
        map_i = item_map[row.item_id]
        mr_dense[map_u, map_i] = row.rating

    user_target = []
    for row in read_test.itertuples():
        if row.user_id in user_map:
            user_target.append(user_map[row.user_id])
    user_target = sorted(list(set(user_target)))

    # item target untuk prior: gunakan semua item dalam fold
    item_target = list(range(b_item_dense))

    return mr_global, mr_dense, read_test, user_map, item_map, user_target, item_target

In [55]:
# load_data(fold=2)

## Eksperimen

In [64]:
import math
import numpy as np
import pandas as pd
import os
import json
from joblib import Parallel, delayed

LIKE_COLUMNS = [
    "fold",
    "k_user",
    "k_item",
    "user_id",
    "item_id",
    "rating_value",
    "likelihood_user",
    "likelihood_item",
]

POST_COLUMNS = [
    "fold",
    "k_user",
    "k_item",
    "user_id",
    "item_id",
    "rating_value",
    "posterior_user",
    "posterior_item",
    "score_user",
    "score_item",
    "score_hybrid",
]

PRED_COLUMNS = [
    "fold",
    "k_user",
    "k_item",
    "user_id",
    "item_id",
    "predicted_rating",
]

TOPN_COLUMNS = [
    "fold",
    "k_user",
    "k_item",
    "user_id",
    "rank",
    "item_id",
    "predicted_rating",
]


def precompute_fold_assets(fold, alpha, R, output_dir):
    print(f"--- Precompute Fold {fold} ---")
    mr_global, mr_dense, df_test, user_map, item_map, user_target, item_target = load_data(fold)

    itr_dir = os.path.join(output_dir, "itr", f"fold_{fold}")
    prior_dir = os.path.join(output_dir, "prior", f"fold_{fold}")

    prior_ub_path = os.path.join(prior_dir, "prior_ub.npy")
    prior_ib_path = os.path.join(prior_dir, "prior_ib.npy")
    user_target_path = os.path.join(prior_dir, "user_target.json")
    item_target_path = os.path.join(prior_dir, "item_target.json")
    itr_user_path = os.path.join(itr_dir, "ITR_user.npy")
    itr_item_path = os.path.join(itr_dir, "ITR_item.npy")

    if (
        os.path.exists(prior_ub_path)
        and os.path.exists(prior_ib_path)
        and os.path.exists(user_target_path)
        and os.path.exists(item_target_path)
        and os.path.exists(itr_user_path)
        and os.path.exists(itr_item_path)
    ):
        return f"Fold {fold} precompute skip (cached)"

    prior_ub, prior_ib = prior_prob(
        mr_dense, target_user=user_target, target_item=item_target, alpha=alpha, R=R
    )

    global mean_user, mean_item
    mean_user = compute_mean_rating_user(mr_global)
    mean_item = compute_mean_rating_item(mr_global)
    mcru = compute_mean_centered_rating_user(mr_global)
    mcri = compute_mean_centered_rating_item(mr_global)
    sdu = standar_deviasi_user(mr_global, mcru)
    sdi = standar_deviasi_item(mr_global, mcri)

    triangle_user = triangle(mr_global, base="user")
    triangle_item = triangle(mr_global, base="item")
    urp_user = URP(mr_global, mean_user, mean_item, sdu, sdi, base="user")
    urp_item = URP(mr_global, mean_user, mean_item, sdu, sdi, base="item")
    ITR_user = ITR(mr_global, triangle_user, triangle_item, urp_user, urp_item, base="user")
    ITR_item = ITR(mr_global, triangle_user, triangle_item, urp_user, urp_item, base="item")

    os.makedirs(itr_dir, exist_ok=True)
    os.makedirs(prior_dir, exist_ok=True)
    np.save(itr_user_path, ITR_user)
    np.save(itr_item_path, ITR_item)
    np.save(prior_ub_path, prior_ub)
    np.save(prior_ib_path, prior_ib)
    with open(user_target_path, "w", encoding="utf-8") as f:
        json.dump(user_target, f)
    with open(item_target_path, "w", encoding="utf-8") as f:
        json.dump(item_target, f)

    return f"Fold {fold} precompute selesai"


def load_fold_assets(fold, output_dir):
    itr_dir = os.path.join(output_dir, "itr", f"fold_{fold}")
    prior_dir = os.path.join(output_dir, "prior", f"fold_{fold}")

    prior_ub_path = os.path.join(prior_dir, "prior_ub.npy")
    prior_ib_path = os.path.join(prior_dir, "prior_ib.npy")
    user_target_path = os.path.join(prior_dir, "user_target.json")
    item_target_path = os.path.join(prior_dir, "item_target.json")
    itr_user_path = os.path.join(itr_dir, "ITR_user.npy")
    itr_item_path = os.path.join(itr_dir, "ITR_item.npy")

    prior_ub = np.load(prior_ub_path)
    prior_ib = np.load(prior_ib_path)
    ITR_user = np.load(itr_user_path)
    ITR_item = np.load(itr_item_path)

    with open(user_target_path, "r", encoding="utf-8") as f:
        user_target = json.load(f)
    with open(item_target_path, "r", encoding="utf-8") as f:
        item_target = json.load(f)

    prior_u_map = {item_dense: i for i, item_dense in enumerate(item_target)}
    prior_i_map = {user_dense: i for i, user_dense in enumerate(user_target)}

    return prior_ub, prior_ib, ITR_user, ITR_item, prior_u_map, prior_i_map


def run_likelihood_combo(fold, k_user, k_item, alpha, R, output_dir):
    print(f"--- Likelihood Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    mr_global, mr_dense, df_test, user_map, item_map, user_target, item_target = load_data(fold)
    prior_ub, prior_ib, ITR_user, ITR_item, prior_u_map, prior_i_map = load_fold_assets(
        fold, output_dir
    )

    grouped = list(df_test.groupby("user_id"))
    item_map_inv = {idx: item_id for item_id, idx in item_map.items()}

    like_dir = os.path.join(output_dir, "likelihood", f"fold_{fold}")
    os.makedirs(like_dir, exist_ok=True)

    like_path = os.path.join(
        like_dir, f"likelihood_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    if os.path.exists(like_path):
        return None

    for user_id, group in grouped:
        user_dense = user_map.get(user_id)
        if user_dense is None:
            continue

        unrated_dense = np.where(mr_dense[user_dense, :] == 0)[0]
        user_like_rows = []

        for item_dense in unrated_dense:
            item_id = item_map_inv[item_dense]
            pred, _, detail_y = prediksi_rating(
                mr_global,
                user_id,
                item_id,
                user_map,
                item_map,
                prior_ub,
                prior_ib,
                ITR_user,
                ITR_item,
                prior_u_map=prior_u_map,
                prior_i_map=prior_i_map,
                k_user=k_user,
                k_item=k_item,
                R=R,
            )
            if pred is None:
                continue

            for detail in detail_y:
                user_like_rows.append(
                    {
                        "fold": fold,
                        "k_user": k_user,
                        "k_item": k_item,
                        "user_id": user_id,
                        "item_id": item_id,
                        "rating_value": detail["rating_value"],
                        "likelihood_user": detail["likelihood_user"],
                        "likelihood_item": detail["likelihood_item"],
                    }
                )

        if user_like_rows:
            with open(like_path, "ab") as f:
                pickle.dump(pd.DataFrame(user_like_rows), f)

    if not os.path.exists(like_path):
        with open(like_path, "ab") as f:
            pickle.dump(pd.DataFrame(columns=LIKE_COLUMNS), f)

    return None


def run_posterior_combo(fold, k_user, k_item, R, output_dir):
    print(f"--- Posterior Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    _, _, _, user_map, item_map, _, _ = load_data(fold)
    prior_ub, prior_ib, _, _, prior_u_map, prior_i_map = load_fold_assets(fold, output_dir)

    like_dir = os.path.join(output_dir, "likelihood", f"fold_{fold}")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    os.makedirs(post_dir, exist_ok=True)

    like_path = os.path.join(
        like_dir, f"likelihood_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    if os.path.exists(post_path):
        return None
    if not os.path.exists(like_path):
        print(f"  File tidak ditemukan: {like_path}")
        return None

    data_frames = []
    with open(like_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_like = pd.concat(data_frames, ignore_index=True)
    else:
        df_like = pd.DataFrame()

    if df_like.empty:
        with open(post_path, "ab") as f:
            pickle.dump(pd.DataFrame(columns=POST_COLUMNS), f)
        return None

    item_dense = df_like["item_id"].map(item_map)
    user_dense = df_like["user_id"].map(user_map)
    idx_prior_user = item_dense.map(prior_u_map)
    idx_prior_item = user_dense.map(prior_i_map)
    valid_mask = idx_prior_user.notna() & idx_prior_item.notna()

    if not valid_mask.any():
        with open(post_path, "ab") as f:
            pickle.dump(pd.DataFrame(columns=POST_COLUMNS), f)
        return None

    df_like = df_like.loc[valid_mask].copy()
    y_index = df_like["rating_value"].to_numpy() - 1
    idx_prior_user = idx_prior_user[valid_mask].astype(int).to_numpy()
    idx_prior_item = idx_prior_item[valid_mask].astype(int).to_numpy()

    prior_u = prior_ub[y_index, idx_prior_user]
    prior_i = prior_ib[y_index, idx_prior_item]
    like_u = df_like["likelihood_user"].to_numpy()
    like_i = df_like["likelihood_item"].to_numpy()

    posterior_u = prior_u * like_u
    posterior_i = prior_i * like_i

    if k_item > 0:
        score_ub = posterior_u ** (1.0 / (k_item + 1))
    else:
        score_ub = np.full_like(posterior_u, -100.0, dtype=float)

    if k_user > 0:
        score_ib = posterior_i ** (1.0 / (k_user + 1))
    else:
        score_ib = np.full_like(posterior_i, -100.0, dtype=float)

    score_hybrid = score_ub * score_ib

    df_post = pd.DataFrame(
        {
            "user_id": df_like["user_id"].to_numpy(),
            "item_id": df_like["item_id"].to_numpy(),
            "rating_value": df_like["rating_value"].to_numpy(),
            "score_user": score_ub,
            "score_item": score_ib,
            "score_hybrid": score_hybrid,
        }
    )
    with open(post_path, "ab") as f:
        pickle.dump(df_post, f)
    return None


def run_prediksi_combo_kr(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_hb_kr", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}_kitem_{k_item}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None
    
    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
            pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
            return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_hybrid"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        if len(idx) == 1:
            pred = float(ratings[idx[0]])
        else:
            Z = float(np.sum(scores))
            if Z == 0.0:
                pred = float(ratings[idx[0]])
            else:
                pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_EV(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_hb_EV", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}_kitem_{k_item}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_hybrid"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        Z = float(np.sum(scores))
        if Z == 0.0:
            pred = float(ratings[idx[0]])
        else:
            pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_ubkr(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_ub_kr", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kitem_{k_item}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_user"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        if len(idx) == 1:
            pred = float(ratings[idx[0]])
        else:
            Z = float(np.sum(scores))
            if Z == 0.0:
                pred = float(ratings[idx[0]])
            else:
                pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_ubEV(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_ub_EV", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kitem_{k_item}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_user"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        Z = float(np.sum(scores))
        if Z == 0.0:
            pred = float(ratings[idx[0]])
        else:
            pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_ibkr(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_ib_kr", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_item"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        if len(idx) == 1:
            pred = float(ratings[idx[0]])
        else:
            Z = float(np.sum(scores))
            if Z == 0.0:
                pred = float(ratings[idx[0]])
            else:
                pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_ibEV(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_ib_EV", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_item"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        Z = float(np.sum(scores))
        if Z == 0.0:
            pred = float(ratings[idx[0]])
        else:
            pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None


def run_topn_combo_kr(fold, k_user, k_item, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    pred_dir = os.path.join(output_dir, "prediksi_hb_kr", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_hb_kr", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}_kitem_{k_item}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kuser_{k_user}_kitem_{k_item}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ev(fold, k_user, k_item, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    pred_dir = os.path.join(output_dir, "prediksi_hb_EV", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_hb_EV", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}_kitem_{k_item}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kuser_{k_user}_kitem_{k_item}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ubkr(fold, k_item, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_item={k_item} ---")
    pred_dir = os.path.join(output_dir, "prediksi_ub_kr", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_ub_kr", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kitem_{k_item}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kitem_{k_item}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ubev(fold, k_item, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_item={k_item} ---")
    pred_dir = os.path.join(output_dir, "prediksi_ub_EV", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_EV", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kitem_{k_item}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kitem_{k_item}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ibkr(fold, k_user, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_user={k_user} ---")
    pred_dir = os.path.join(output_dir, "prediksi_ib_kr", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_ib_kr", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kuser_{k_user}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ibev(fold, k_user, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_user={k_user}, ---")
    pred_dir = os.path.join(output_dir, "prediksi_ib_EV", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_ib_EV", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kuser_{k_user}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_cv_5fold_grid(
    k_values=None,
    k_folds=5,
    alpha=0.01,
    R=5,
    top_n=100,
    output_dir="hasil_eval_ndcg",
    n_jobs=8,
    precompute_jobs=None,
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    if precompute_jobs is None:
        precompute_jobs = n_jobs

    print(
        f"Inisialisasi uji {k_folds}-fold CV (staged, n_jobs={n_jobs}, precompute_jobs={precompute_jobs})..."
    )
    os.makedirs(output_dir, exist_ok=True)

    print("Tahap A: precompute prior dan ITR per fold...")
    Parallel(n_jobs=precompute_jobs, verbose=10)(
        delayed(precompute_fold_assets)(fold, alpha, R, output_dir)
        for fold in range(1, k_folds + 1)
    )

    print("Tahap B1: Likelihood parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_likelihood_combo)(fold, k_user, k_item, alpha, R, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B2: Posterior parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_posterior_combo)(fold, k_user, k_item, R, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3A: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_kr)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3B: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_EV)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3C: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_ubkr)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3D: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_ubEV)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3E: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_ibkr)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3F: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_ibEV)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B4A: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_kr)(fold, k_user, k_item, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )
    print("Tahap B4B: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ev)(fold, k_user, k_item, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )
    print("Tahap B4C: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ubkr)(fold, k_item, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )
    print("Tahap B4D: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ubev)(fold, k_item, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B4E: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ibkr)(fold, k_user, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )
    print("Tahap B4F: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ibev)(fold, k_user, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Selesai CV staged.")


def eval_ndcg_from_saved_hbkr(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_kr'
    save_dir = os.path.join(output_dir, "ndcg_hybrid_kr")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_user in k_values:
        for k_item in k_values:
            fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
            all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

            for fold in range(1, k_folds + 1):
                # Tetap membaca top-N yang tersimpan dari folder topn_kr
                topn_path = os.path.join(
                    output_dir,
                    "topn_hb_kr",
                    f"fold_{fold}",
                    f"topn_{k_max}_kuser_{k_user}_kitem_{k_item}.csv",
                )
                if not os.path.exists(topn_path):
                    print(f"  File tidak ditemukan: {topn_path}")
                    for k_eval in range(1, k_max + 1):
                        fold_scores_per_k[k_eval].append(0.0)
                    continue

                df_topn = pd.read_csv(topn_path)
                if df_topn.empty:
                    for k_eval in range(1, k_max + 1):
                        fold_scores_per_k[k_eval].append(0.0)
                    continue

                gt_map = ground_truth_per_fold[fold]
                user_ranked = {
                    user_id: group.sort_values("rank")["item_id"].to_numpy()
                    for user_id, group in df_topn.groupby("user_id")
                }

                for k_eval in range(1, k_max + 1):
                    user_ndcgs = []
                    for user_id, ranked_items in user_ranked.items():
                        gt_items = gt_map.get(user_id)
                        if not gt_items:
                            continue

                        dcg = 0.0
                        for rank, item_id in enumerate(
                            ranked_items[:k_eval], start=1
                        ):
                            dcg += (1 / math.log2(rank + 1)) * (
                                1 if item_id in gt_items else 0
                            )

                        idcg_len = min(k_eval, len(gt_items))
                        if idcg_len == 0:
                            continue

                        idcg = 0.0
                        for rank in range(1, idcg_len + 1):
                            idcg += 1 / math.log2(rank + 1)

                        ndcg = dcg / idcg if idcg > 0 else 0.0
                        user_ndcgs.append(
                            {
                                "user_id": user_id,
                                "fold": fold,
                                "k_user": k_user,
                                "k_item": k_item,
                                "k": k_eval,
                                "ndcg": ndcg,
                            }
                        )

                    all_user_ndcgs.extend(user_ndcgs)

                    fold_ndcg = (
                        float(np.mean([x["ndcg"] for x in user_ndcgs]))
                        if user_ndcgs
                        else 0.0
                    )
                    fold_scores_per_k[k_eval].append(fold_ndcg)

            # Simpan detail per user ke file CSV di dalam subfolder: ndcg_hybrid_kr/per_user_ndcg/fold_X/
            if all_user_ndcgs:
                per_user_path = os.path.join(
                    per_user_dir, f"k_user_{k_user}_k_item_{k_item}.csv"
                )
                pd.DataFrame(all_user_ndcgs).to_csv(
                    per_user_path, index=False
                )

            print(f"  k_user={k_user}, k_item={k_item}, topn={k_max}")

            for k_eval in range(1, k_max + 1):
                fold_scores = fold_scores_per_k[k_eval]
                mean_ndcg = (
                    float(np.mean(fold_scores)) if fold_scores else 0.0
                )
                results.append(
                    {
                        "k_user": k_user,
                        "k_item": k_item,
                        "k": k_eval,
                        "mean_ndcg": mean_ndcg,
                        "fold_scores": fold_scores,
                    }
                )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

def eval_ndcg_from_saved_hbev(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_EV'
    save_dir = os.path.join(output_dir, "ndcg_hybrid_EV")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_user in k_values:
        for k_item in k_values:
            fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
            all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

            for fold in range(1, k_folds + 1):
                # Tetap membaca top-N yang tersimpan dari folder topn_kr
                topn_path = os.path.join(
                    output_dir,
                    "topn_hb_EV",
                    f"fold_{fold}",
                    f"topn_{k_max}_kuser_{k_user}_kitem_{k_item}.csv",
                )
                if not os.path.exists(topn_path):
                    print(f"  File tidak ditemukan: {topn_path}")
                    for k_eval in range(1, k_max + 1):
                        fold_scores_per_k[k_eval].append(0.0)
                    continue

                df_topn = pd.read_csv(topn_path)
                if df_topn.empty:
                    for k_eval in range(1, k_max + 1):
                        fold_scores_per_k[k_eval].append(0.0)
                    continue

                gt_map = ground_truth_per_fold[fold]
                user_ranked = {
                    user_id: group.sort_values("rank")["item_id"].to_numpy()
                    for user_id, group in df_topn.groupby("user_id")
                }

                for k_eval in range(1, k_max + 1):
                    user_ndcgs = []
                    for user_id, ranked_items in user_ranked.items():
                        gt_items = gt_map.get(user_id)
                        if not gt_items:
                            continue

                        dcg = 0.0
                        for rank, item_id in enumerate(
                            ranked_items[:k_eval], start=1
                        ):
                            dcg += (1 / math.log2(rank + 1)) * (
                                1 if item_id in gt_items else 0
                            )

                        idcg_len = min(k_eval, len(gt_items))
                        if idcg_len == 0:
                            continue

                        idcg = 0.0
                        for rank in range(1, idcg_len + 1):
                            idcg += 1 / math.log2(rank + 1)

                        ndcg = dcg / idcg if idcg > 0 else 0.0
                        user_ndcgs.append(
                            {
                                "user_id": user_id,
                                "fold": fold,
                                "k_user": k_user,
                                "k_item": k_item,
                                "k": k_eval,
                                "ndcg": ndcg,
                            }
                        )

                    all_user_ndcgs.extend(user_ndcgs)

                    fold_ndcg = (
                        float(np.mean([x["ndcg"] for x in user_ndcgs]))
                        if user_ndcgs
                        else 0.0
                    )
                    fold_scores_per_k[k_eval].append(fold_ndcg)

            # Simpan detail per user ke file CSV di dalam subfolder: ndcg_hybrid_kr/per_user_ndcg/fold_X/
            if all_user_ndcgs:
                per_user_path = os.path.join(
                    per_user_dir, f"k_user_{k_user}_k_item_{k_item}.csv"
                )
                pd.DataFrame(all_user_ndcgs).to_csv(
                    per_user_path, index=False
                )

            print(f"  k_user={k_user}, k_item={k_item}, topn={k_max}")

            for k_eval in range(1, k_max + 1):
                fold_scores = fold_scores_per_k[k_eval]
                mean_ndcg = (
                    float(np.mean(fold_scores)) if fold_scores else 0.0
                )
                results.append(
                    {
                        "k_user": k_user,
                        "k_item": k_item,
                        "k": k_eval,
                        "mean_ndcg": mean_ndcg,
                        "fold_scores": fold_scores,
                    }
                )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return 

def eval_ndcg_from_saved_ubkr(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_ub_kr'
    save_dir = os.path.join(output_dir, "ndcg_ub_kr")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_item in k_values:
        fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
        all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

        for fold in range(1, k_folds + 1):
            # Tetap membaca top-N yang tersimpan dari folder topn_kr
            topn_path = os.path.join(
                output_dir,
                "topn_ub_kr",
                f"fold_{fold}",
                f"topn_{k_max}_kitem_{k_item}.csv",
            )
            if not os.path.exists(topn_path):
                print(f"  File tidak ditemukan: {topn_path}")
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            df_topn = pd.read_csv(topn_path)
            if df_topn.empty:
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            gt_map = ground_truth_per_fold[fold]
            user_ranked = {
                user_id: group.sort_values("rank")["item_id"].to_numpy()
                for user_id, group in df_topn.groupby("user_id")
            }

            for k_eval in range(1, k_max + 1):
                user_ndcgs = []
                for user_id, ranked_items in user_ranked.items():
                    gt_items = gt_map.get(user_id)
                    if not gt_items:
                        continue

                    dcg = 0.0
                    for rank, item_id in enumerate(
                        ranked_items[:k_eval], start=1
                    ):
                        dcg += (1 / math.log2(rank + 1)) * (
                            1 if item_id in gt_items else 0
                        )

                    idcg_len = min(k_eval, len(gt_items))
                    if idcg_len == 0:
                        continue

                    idcg = 0.0
                    for rank in range(1, idcg_len + 1):
                        idcg += 1 / math.log2(rank + 1)

                    ndcg = dcg / idcg if idcg > 0 else 0.0
                    user_ndcgs.append(
                        {
                            "user_id": user_id,
                            "fold": fold,
                            "k_item": k_item,
                            "k": k_eval,
                            "ndcg": ndcg,
                        }
                    )

                all_user_ndcgs.extend(user_ndcgs)

                fold_ndcg = (
                    float(np.mean([x["ndcg"] for x in user_ndcgs]))
                    if user_ndcgs
                    else 0.0
                )
                fold_scores_per_k[k_eval].append(fold_ndcg)

        # Simpan detail per user ke file CSV di dalam subfolder: ndcg_ub_kr/per_user_ndcg/fold_X/
        if all_user_ndcgs:
            per_user_path = os.path.join(
                per_user_dir, f"k_item_{k_item}.csv"
            )
            pd.DataFrame(all_user_ndcgs).to_csv(
                per_user_path, index=False
            )

        print(f"  k_item={k_item}, topn={k_max}")

        for k_eval in range(1, k_max + 1):
            fold_scores = fold_scores_per_k[k_eval]
            mean_ndcg = (
                float(np.mean(fold_scores)) if fold_scores else 0.0
            )
            results.append(
                {
                    "k_item": k_item,
                    "k": k_eval,
                    "mean_ndcg": mean_ndcg,
                    "fold_scores": fold_scores,
                }
            )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

def eval_ndcg_from_saved_ibkr(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_kr'
    save_dir = os.path.join(output_dir, "ndcg_ib_kr")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_user in k_values:
        fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
        all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

        for fold in range(1, k_folds + 1):
            # Tetap membaca top-N yang tersimpan dari folder topn_kr
            topn_path = os.path.join(
                output_dir,
                "topn_ib_kr",
                f"fold_{fold}",
                f"topn_{k_max}_kuser_{k_user}.csv",
            )
            if not os.path.exists(topn_path):
                print(f"  File tidak ditemukan: {topn_path}")
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            df_topn = pd.read_csv(topn_path)
            if df_topn.empty:
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            gt_map = ground_truth_per_fold[fold]
            user_ranked = {
                user_id: group.sort_values("rank")["item_id"].to_numpy()
                for user_id, group in df_topn.groupby("user_id")
            }

            for k_eval in range(1, k_max + 1):
                user_ndcgs = []
                for user_id, ranked_items in user_ranked.items():
                    gt_items = gt_map.get(user_id)
                    if not gt_items:
                        continue

                    dcg = 0.0
                    for rank, item_id in enumerate(
                        ranked_items[:k_eval], start=1
                    ):
                        dcg += (1 / math.log2(rank + 1)) * (
                            1 if item_id in gt_items else 0
                        )

                    idcg_len = min(k_eval, len(gt_items))
                    if idcg_len == 0:
                        continue

                    idcg = 0.0
                    for rank in range(1, idcg_len + 1):
                        idcg += 1 / math.log2(rank + 1)

                    ndcg = dcg / idcg if idcg > 0 else 0.0
                    user_ndcgs.append(
                        {
                            "user_id": user_id,
                            "fold": fold,
                            "k_user": k_user,
                            "k": k_eval,
                            "ndcg": ndcg,
                        }
                    )

                all_user_ndcgs.extend(user_ndcgs)

                fold_ndcg = (
                    float(np.mean([x["ndcg"] for x in user_ndcgs]))
                    if user_ndcgs
                    else 0.0
                )
                fold_scores_per_k[k_eval].append(fold_ndcg)

        # Simpan detail per user ke file CSV di dalam subfolder: ndcg_ib_kr/per_user_ndcg/fold_X/
        if all_user_ndcgs:
            per_user_path = os.path.join(
                per_user_dir, f"k_user_{k_user}_.csv"
            )
            pd.DataFrame(all_user_ndcgs).to_csv(
                per_user_path, index=False
            )

        print(f"  k_user={k_user},topn={k_max}")

        for k_eval in range(1, k_max + 1):
            fold_scores = fold_scores_per_k[k_eval]
            mean_ndcg = (
                float(np.mean(fold_scores)) if fold_scores else 0.0
            )
            results.append(
                {
                    "k_user": k_user,
                    "k": k_eval,
                    "mean_ndcg": mean_ndcg,
                    "fold_scores": fold_scores,
                }
            )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

def eval_ndcg_from_saved_ubev(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_kr'
    save_dir = os.path.join(output_dir, "ndcg_ub_ev")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_item in k_values:
        fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
        all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

        for fold in range(1, k_folds + 1):
            # Tetap membaca top-N yang tersimpan dari folder topn_kr
            topn_path = os.path.join(
                output_dir,
                "topn_ub_EV",
                f"fold_{fold}",
                f"topn_{k_max}_kitem_{k_item}.csv",
            )
            if not os.path.exists(topn_path):
                print(f"  File tidak ditemukan: {topn_path}")
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            df_topn = pd.read_csv(topn_path)
            if df_topn.empty:
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            gt_map = ground_truth_per_fold[fold]
            user_ranked = {
                user_id: group.sort_values("rank")["item_id"].to_numpy()
                for user_id, group in df_topn.groupby("user_id")
            }

            for k_eval in range(1, k_max + 1):
                user_ndcgs = []
                for user_id, ranked_items in user_ranked.items():
                    gt_items = gt_map.get(user_id)
                    if not gt_items:
                        continue

                    dcg = 0.0
                    for rank, item_id in enumerate(
                        ranked_items[:k_eval], start=1
                    ):
                        dcg += (1 / math.log2(rank + 1)) * (
                            1 if item_id in gt_items else 0
                        )

                    idcg_len = min(k_eval, len(gt_items))
                    if idcg_len == 0:
                        continue

                    idcg = 0.0
                    for rank in range(1, idcg_len + 1):
                        idcg += 1 / math.log2(rank + 1)

                    ndcg = dcg / idcg if idcg > 0 else 0.0
                    user_ndcgs.append(
                        {
                            "user_id": user_id,
                            "fold": fold,
                            "k_item": k_item,
                            "k": k_eval,
                            "ndcg": ndcg,
                        }
                    )

                all_user_ndcgs.extend(user_ndcgs)

                fold_ndcg = (
                    float(np.mean([x["ndcg"] for x in user_ndcgs]))
                    if user_ndcgs
                    else 0.0
                )
                fold_scores_per_k[k_eval].append(fold_ndcg)

        # Simpan detail per user ke file CSV di dalam subfolder: ndcg_hybrid_kr/per_user_ndcg/fold_X/
        if all_user_ndcgs:
            per_user_path = os.path.join(
                per_user_dir, f"k_item_{k_item}.csv"
            )
            pd.DataFrame(all_user_ndcgs).to_csv(
                per_user_path, index=False
            )

        print(f"  k_item={k_item}, topn={k_max}")

        for k_eval in range(1, k_max + 1):
            fold_scores = fold_scores_per_k[k_eval]
            mean_ndcg = (
                float(np.mean(fold_scores)) if fold_scores else 0.0
            )
            results.append(
                {
                    "k_item": k_item,
                    "k": k_eval,
                    "mean_ndcg": mean_ndcg,
                    "fold_scores": fold_scores,
                }
            )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

def eval_ndcg_from_saved_ibev(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_kr'
    save_dir = os.path.join(output_dir, "ndcg_ib_ev")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_user in k_values:
        fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
        all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

        for fold in range(1, k_folds + 1):
            # Tetap membaca top-N yang tersimpan dari folder topn_kr
            topn_path = os.path.join(
                output_dir,
                "topn_ib_EV",
                f"fold_{fold}",
                f"topn_{k_max}_kuser_{k_user}.csv",
            )
            if not os.path.exists(topn_path):
                print(f"  File tidak ditemukan: {topn_path}")
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            df_topn = pd.read_csv(topn_path)
            if df_topn.empty:
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            gt_map = ground_truth_per_fold[fold]
            user_ranked = {
                user_id: group.sort_values("rank")["item_id"].to_numpy()
                for user_id, group in df_topn.groupby("user_id")
            }

            for k_eval in range(1, k_max + 1):
                user_ndcgs = []
                for user_id, ranked_items in user_ranked.items():
                    gt_items = gt_map.get(user_id)
                    if not gt_items:
                        continue

                    dcg = 0.0
                    for rank, item_id in enumerate(
                        ranked_items[:k_eval], start=1
                    ):
                        dcg += (1 / math.log2(rank + 1)) * (
                            1 if item_id in gt_items else 0
                        )

                    idcg_len = min(k_eval, len(gt_items))
                    if idcg_len == 0:
                        continue

                    idcg = 0.0
                    for rank in range(1, idcg_len + 1):
                        idcg += 1 / math.log2(rank + 1)

                    ndcg = dcg / idcg if idcg > 0 else 0.0
                    user_ndcgs.append(
                        {
                            "user_id": user_id,
                            "fold": fold,
                            "k_user": k_user,
                            "k": k_eval,
                            "ndcg": ndcg,
                        }
                    )

                all_user_ndcgs.extend(user_ndcgs)

                fold_ndcg = (
                    float(np.mean([x["ndcg"] for x in user_ndcgs]))
                    if user_ndcgs
                    else 0.0
                )
                fold_scores_per_k[k_eval].append(fold_ndcg)

        # Simpan detail per user ke file CSV di dalam subfolder: ndcg_hybrid_kr/per_user_ndcg/fold_X/
        if all_user_ndcgs:
            per_user_path = os.path.join(
                per_user_dir, f"k_user_{k_user}.csv"
            )
            pd.DataFrame(all_user_ndcgs).to_csv(
                per_user_path, index=False
            )

        print(f"  k_user={k_user}, topn={k_max}")

        for k_eval in range(1, k_max + 1):
            fold_scores = fold_scores_per_k[k_eval]
            mean_ndcg = (
                float(np.mean(fold_scores)) if fold_scores else 0.0
            )
            results.append(
                {
                    "k_user": k_user,
                    "k": k_eval,
                    "mean_ndcg": mean_ndcg,
                    "fold_scores": fold_scores,
                }
            )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

In [57]:
# def best_ndcg_for_k(k=20, output_dir="hasil_eval_ndcg", top_n=5):
#     summary_path = os.path.join(output_dir, "summary_ndcg.csv")
#     if not os.path.exists(summary_path):
#         raise FileNotFoundError(f"Summary tidak ditemukan: {summary_path}")

#     df = pd.read_csv(summary_path)
#     df_k = df[df["k"] == k].copy()
#     if df_k.empty:
#         print(f"Tidak ada data untuk k={k}.")
#         return None

#     df_k = df_k.sort_values(["mean_ndcg", "k_user", "k_item"], ascending=[False, True, True])
#     best_rows = df_k.head(top_n)
#     print(best_rows[["k_user", "k_item", "mean_ndcg"]])
#     return best_rows

In [58]:
# def dcg(ground_truth, top_n, n):
#     a = np.array([(1 / np.log2(1 + x)) for x in range(1, n + 1)])
#     b = np.array([np.sum(np.where(tp == ground_truth, 1, 0)) for tp in top_n[:n]])
#     return np.sum(a * b)

In [59]:
# run_likelihood_stage(fold=5, 
#                      k_values=[10,20,30,40,50,60,70,80,90,100], 
#                      alpha=0.01, 
#                      R=5,
#                      output_dir="hasil_eval_ndcg",
#                      n_jobs=12)

In [62]:
run_cv_5fold_grid(n_jobs=20)

Inisialisasi uji 5-fold CV (staged, n_jobs=20, precompute_jobs=20)...
Tahap A: precompute prior dan ITR per fold...


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   2 out of   5 | elapsed:    0.9s remaining:    1.4s
[Parallel(n_jobs=20)]: Done   3 out of   5 | elapsed:    0.9s remaining:    0.6s
[Parallel(n_jobs=20)]: Done   5 out of   5 | elapsed:    0.9s finished
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.


Tahap B1: Likelihood parallel untuk semua kombinasi...


[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    1.0s
[Parallel(n_jobs=20)]: Done  21 tasks      | elapsed:    1.4s
[Parallel(n_jobs=20)]: Done  32 tasks      | elapsed:    1.5s
[Parallel(n_jobs=20)]: Done  45 tasks      | elapsed:    1.8s
[Parallel(n_jobs=20)]: Done  58 tasks      | elapsed:    1.9s
[Parallel(n_jobs=20)]: Done  73 tasks      | elapsed:    2.3s
[Parallel(n_jobs=20)]: Done  88 tasks      | elapsed:    2.6s
[Parallel(n_jobs=20)]: Done 105 tasks      | elapsed:    3.0s
[Parallel(n_jobs=20)]: Done 122 tasks      | elapsed:    3.3s
[Parallel(n_jobs=20)]: Done 141 tasks      | elapsed:    3.7s
[Parallel(n_jobs=20)]: Done 160 tasks      | elapsed:    4.0s
[Parallel(n_jobs=20)]: Done 181 tasks      | elapsed:    4.5s
[Parallel(n_jobs=20)]: Done 202 tasks      | elapsed:    4.9s
[Parallel(n_jobs=20)]: Done 225 tasks      | elapsed:    5.5s
[Parallel(n_jobs=20)]: Done 248 tasks      | elapsed:    5.9s
[Paralle

Tahap B2: Posterior parallel untuk semua kombinasi...


...

[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    2.4s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    2.6s
[Parallel(n_jobs=20)]: Done  21 tasks      | elapsed:    2.8s
[Parallel(n_jobs=20)]: Done  32 tasks      | elapsed:    5.4s
[Parallel(n_jobs=20)]: Done  45 tasks      | elapsed:    7.7s
[Parallel(n_jobs=20)]: Done  58 tasks      | elapsed:    8.3s
[Parallel(n_jobs=20)]: Done  73 tasks      | elapsed:   12.3s
[Parallel(n_jobs=20)]: Done  88 tasks      | elapsed:   16.4s
[Parallel(n_jobs=20)]: Done 105 tasks      | elapsed:   20.2s
[Parallel(n_jobs=20)]: Done 122 tasks      | elapsed:   21.1s
[Parallel(n_jobs=20)]: Done 141 tasks      | elapsed:   26.0s
[Parallel(n_jobs=20)]: Done 160 tasks      | elapsed:   31.0s
[Parallel(n_jobs=20)]: Done 181 tasks      | elapsed:   36.2s
[Parallel(n_jobs=20)]: Done 202 tasks      | elapsed:   41.6s
[Parallel(n_jobs=20)]: Done 225 tasks      | elapsed:   49.3s
[Parallel(n_jobs=20)]: Done 248 tasks      | elapsed:   55.4s
[Paralle

Tahap B4F: TopN parallel untuk semua kombinasi...


[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    2.7s
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    2.8s
[Parallel(n_jobs=20)]: Done  21 tasks      | elapsed:    2.9s
[Parallel(n_jobs=20)]: Done  32 tasks      | elapsed:    5.7s
[Parallel(n_jobs=20)]: Done  45 tasks      | elapsed:    8.4s
[Parallel(n_jobs=20)]: Done  58 tasks      | elapsed:    8.8s
[Parallel(n_jobs=20)]: Done  73 tasks      | elapsed:   12.8s
[Parallel(n_jobs=20)]: Done  88 tasks      | elapsed:   16.7s
[Parallel(n_jobs=20)]: Done 105 tasks      | elapsed:   20.5s
[Parallel(n_jobs=20)]: Done 122 tasks      | elapsed:   21.3s
[Parallel(n_jobs=20)]: Done 141 tasks      | elapsed:   26.7s
[Parallel(n_jobs=20)]: Done 160 tasks      | elapsed:   32.2s
[Parallel(n_jobs=20)]: Done 181 tasks      | elapsed:   37.9s
[Parallel(n_jobs=20)]: Done 202 tasks      | elapsed:   43.9s
[Parallel(n_jobs=20)]: Done 225 tasks      | elapsed:   53.0s
[Parallel(n_jobs=20)]: Done 248 tasks      | elapsed:   59.0s
[Paralle

Selesai CV staged.


[Parallel(n_jobs=20)]: Done 320 out of 320 | elapsed:  1.3min finished


In [65]:
eval_ndcg_from_saved_hbkr(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_user=10, k_item=10, topn=100
  k_user=10, k_item=20, topn=100
  k_user=10, k_item=30, topn=100
  k_user=10, k_item=40, topn=100
  k_user=10, k_item=50, topn=100
  k_user=10, k_item=100, topn=100
  k_user=10, k_item=150, topn=100
  k_user=10, k_item=200, topn=100
  k_user=20, k_item=10, topn=100
  k_user=20, k_item=20, topn=100
  k_user=20, k_item=30, topn=100
  k_user=20, k_item=40, topn=100
  k_user=20, k_item=50, topn=100
  k_user=20, k_item=100, topn=100
  k_user=20, k_item=150, topn=100
  k_user=20, k_item=200, topn=100
  k_user=30, k_item=10, topn=100
  k_user=30, k_item=20, topn=100
  k_user=30, k_item=30, topn=100
  k_user=30, k_item=40, topn=100
  k_user=30, k_item=50, topn=100
  k_user=30, k_item=100, topn=100
  k_user=30, k_item=150, topn=100
  k_user=30, k_item=200, topn=100
  k_user=40, k_item=10, topn=100
  k_user=40, k_item=20, topn=100
  k_user=40, k_item=30, topn=100
  k_user=40, k_item=40, topn=100
  k_user=40, k_ite

[{'k_user': 10,
  'k_item': 10,
  'k': 1,
  'mean_ndcg': 0.11138369776243244,
  'fold_scores': [0.14814814814814814,
   0.13476263399693722,
   0.09666283084004602,
   0.09967497291440953,
   0.07766990291262135]},
 {'k_user': 10,
  'k_item': 10,
  'k': 2,
  'mean_ndcg': 0.09977989214059935,
  'fold_scores': [0.1338202663987207,
   0.12054446037729098,
   0.08954010941800614,
   0.08149789103793834,
   0.07349673347104055]},
 {'k_user': 10,
  'k_item': 10,
  'k': 3,
  'mean_ndcg': 0.09527772635689112,
  'fold_scores': [0.13258137962195618,
   0.11346017244944757,
   0.08257103545245344,
   0.07550057408086783,
   0.07227547017973063]},
 {'k_user': 10,
  'k_item': 10,
  'k': 4,
  'mean_ndcg': 0.09118231743250685,
  'fold_scores': [0.12823905441675795,
   0.10880268114036946,
   0.07860106055390466,
   0.07075436831865196,
   0.06951442273285018]},
 {'k_user': 10,
  'k_item': 10,
  'k': 5,
  'mean_ndcg': 0.0895239559748148,
  'fold_scores': [0.128833745217524,
   0.10533320283528685,
   

In [66]:
eval_ndcg_from_saved_hbev(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_user=10, k_item=10, topn=100
  k_user=10, k_item=20, topn=100
  k_user=10, k_item=30, topn=100
  k_user=10, k_item=40, topn=100
  k_user=10, k_item=50, topn=100
  k_user=10, k_item=100, topn=100
  k_user=10, k_item=150, topn=100
  k_user=10, k_item=200, topn=100
  k_user=20, k_item=10, topn=100
  k_user=20, k_item=20, topn=100
  k_user=20, k_item=30, topn=100
  k_user=20, k_item=40, topn=100
  k_user=20, k_item=50, topn=100
  k_user=20, k_item=100, topn=100
  k_user=20, k_item=150, topn=100
  k_user=20, k_item=200, topn=100
  k_user=30, k_item=10, topn=100
  k_user=30, k_item=20, topn=100
  k_user=30, k_item=30, topn=100
  k_user=30, k_item=40, topn=100
  k_user=30, k_item=50, topn=100
  k_user=30, k_item=100, topn=100
  k_user=30, k_item=150, topn=100
  k_user=30, k_item=200, topn=100
  k_user=40, k_item=10, topn=100
  k_user=40, k_item=20, topn=100
  k_user=40, k_item=30, topn=100
  k_user=40, k_item=40, topn=100
  k_user=40, k_ite

In [67]:
eval_ndcg_from_saved_ubkr(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_item=10, topn=100
  k_item=20, topn=100
  k_item=30, topn=100
  k_item=40, topn=100
  k_item=50, topn=100
  k_item=100, topn=100
  k_item=150, topn=100
  k_item=200, topn=100
Detail per user disimpan di folder: hasil_eval_ndcg\ndcg_ub_kr\per_user_ndcg\fold_5
Ringkasan evaluasi disimpan di: hasil_eval_ndcg\ndcg_ub_kr\summary_ndcg.csv


[{'k_item': 10,
  'k': 1,
  'mean_ndcg': 0.09444245566935427,
  'fold_scores': [0.11328976034858387,
   0.11485451761102604,
   0.08515535097813579,
   0.0780065005417118,
   0.08090614886731391]},
 {'k_item': 10,
  'k': 2,
  'mean_ndcg': 0.08688732684732618,
  'fold_scores': [0.11834665978955827,
   0.1077454308012029,
   0.07536160902283093,
   0.06501361102462537,
   0.06796932359841339]},
 {'k_item': 10,
  'k': 3,
  'mean_ndcg': 0.0862171074690948,
  'fold_scores': [0.12176088394966796,
   0.10402362423160634,
   0.07279941245632468,
   0.06399483310062418,
   0.06850678360725096]},
 {'k_item': 10,
  'k': 4,
  'mean_ndcg': 0.08433855059890624,
  'fold_scores': [0.11857923025333918,
   0.10532965679575536,
   0.07034385122390066,
   0.06106813325811284,
   0.06637188146342315]},
 {'k_item': 10,
  'k': 5,
  'mean_ndcg': 0.0821326535581418,
  'fold_scores': [0.11748288768905565,
   0.10123278611919355,
   0.06754376404100312,
   0.05919347782707542,
   0.06521035211438125]},
 {'k_item

In [68]:
eval_ndcg_from_saved_ubev(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_item=10, topn=100
  k_item=20, topn=100
  k_item=30, topn=100
  k_item=40, topn=100
  k_item=50, topn=100
  k_item=100, topn=100
  k_item=150, topn=100
  k_item=200, topn=100
Detail per user disimpan di folder: hasil_eval_ndcg\ndcg_ub_ev\per_user_ndcg\fold_5
Ringkasan evaluasi disimpan di: hasil_eval_ndcg\ndcg_ub_ev\summary_ndcg.csv


[{'k_item': 10,
  'k': 1,
  'mean_ndcg': 0.0976459934090744,
  'fold_scores': [0.13725490196078433,
   0.10107197549770292,
   0.09205983889528194,
   0.08017334777898158,
   0.07766990291262135]},
 {'k_item': 10,
  'k': 2,
  'mean_ndcg': 0.09584869147023745,
  'fold_scores': [0.13725490196078433,
   0.10047955159688433,
   0.09161466880640444,
   0.07556296762775737,
   0.07433136735935672]},
 {'k_item': 10,
  'k': 3,
  'mean_ndcg': 0.09571324863952065,
  'fold_scores': [0.13367652387564347,
   0.10421180967710247,
   0.09018183675988173,
   0.07715317194517686,
   0.07334290093979874]},
 {'k_item': 10,
  'k': 4,
  'mean_ndcg': 0.09426343130261507,
  'fold_scores': [0.13098152940868754,
   0.10406221144490282,
   0.08737826640087702,
   0.07574274057518139,
   0.07315240868342655]},
 {'k_item': 10,
  'k': 5,
  'mean_ndcg': 0.0929923930682644,
  'fold_scores': [0.12625501601261194,
   0.10529314304380649,
   0.08576212517072382,
   0.0725249522757104,
   0.07512672883846934]},
 {'k_ite

In [69]:
eval_ndcg_from_saved_ibkr(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_user=10,topn=100
  k_user=20,topn=100
  k_user=30,topn=100
  k_user=40,topn=100
  k_user=50,topn=100
  k_user=100,topn=100
  k_user=150,topn=100
  k_user=200,topn=100
Detail per user disimpan di folder: hasil_eval_ndcg\ndcg_ib_kr\per_user_ndcg\fold_5
Ringkasan evaluasi disimpan di: hasil_eval_ndcg\ndcg_ib_kr\summary_ndcg.csv


[{'k_user': 10,
  'k': 1,
  'mean_ndcg': 0.111690069372431,
  'fold_scores': [0.14814814814814814,
   0.13935681470137826,
   0.1001150747986191,
   0.08992416034669556,
   0.08090614886731391]},
 {'k_user': 10,
  'k': 2,
  'mean_ndcg': 0.1005066574456042,
  'fold_scores': [0.1439340652806695,
   0.11862197817272747,
   0.08631480204341682,
   0.076512145361316,
   0.0771502963698912]},
 {'k_user': 10,
  'k': 3,
  'mean_ndcg': 0.09682440432050712,
  'fold_scores': [0.14032208317661932,
   0.1152227071469112,
   0.08172257679255258,
   0.07279535866498928,
   0.07405929582146333]},
 {'k_user': 10,
  'k': 4,
  'mean_ndcg': 0.09308326770594125,
  'fold_scores': [0.135410914411161,
   0.11052635464923016,
   0.07890800952333477,
   0.06820689479771122,
   0.07236416514826906]},
 {'k_user': 10,
  'k': 5,
  'mean_ndcg': 0.09073517462033845,
  'fold_scores': [0.13049875910038478,
   0.10720362581511991,
   0.07849196231322282,
   0.06766421792467274,
   0.06981730794829198]},
 {'k_user': 10,


In [70]:
eval_ndcg_from_saved_ibev(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_user=10, topn=100
  k_user=20, topn=100
  k_user=30, topn=100
  k_user=40, topn=100
  k_user=50, topn=100
  k_user=100, topn=100
  k_user=150, topn=100
  k_user=200, topn=100
Detail per user disimpan di folder: hasil_eval_ndcg\ndcg_ib_ev\per_user_ndcg\fold_5
Ringkasan evaluasi disimpan di: hasil_eval_ndcg\ndcg_ib_ev\summary_ndcg.csv


[{'k_user': 10,
  'k': 1,
  'mean_ndcg': 0.09316261780966857,
  'fold_scores': [0.13071895424836602,
   0.11026033690658499,
   0.08860759493670886,
   0.07042253521126761,
   0.06580366774541532]},
 {'k_user': 10,
  'k': 2,
  'mean_ndcg': 0.09435183154356552,
  'fold_scores': [0.13830430340982763,
   0.10789064130331062,
   0.08282038378130145,
   0.06916515880638827,
   0.07357867041699964]},
 {'k_user': 10,
  'k': 3,
  'mean_ndcg': 0.09087410439930307,
  'fold_scores': [0.13039011948269336,
   0.10126016094319903,
   0.08228821589915315,
   0.0677586272582818,
   0.07267339841318803]},
 {'k_user': 10,
  'k': 4,
  'mean_ndcg': 0.08949802454529929,
  'fold_scores': [0.1262323967231266,
   0.09788141922812378,
   0.08122248086584102,
   0.06977913181785146,
   0.07237469409155362]},
 {'k_user': 10,
  'k': 5,
  'mean_ndcg': 0.0882691202584562,
  'fold_scores': [0.1244612200918102,
   0.09608984368086446,
   0.07926122557946812,
   0.06984300587209404,
   0.07169030606804416]},
 {'k_user